# titanic3.csv (sample) · version 4

Exported from Sherlock 0.1.0 on 2026-10-25 23:49 UTC. This notebook rebuilds version 4 from the raw file (3 steps), then reruns the dashboard's 10 widgets.

Every number below was computed by code. The cleaning steps were written by NVIDIA Nemotron models on Nebius Token Factory ([models](https://build.nvidia.com/nvidia)) and checked by code before Sherlock kept them; nothing here calls a model.

In [ ]:
DATA_PATH = "titanic3-raw.csv"
RANDOM_SEED = 42


In [ ]:
# The library versions Sherlock used.
%pip install -q pandas==2.2.3 duckdb==1.1.3 numpy==2.1.3 scikit-learn==1.5.2 statsmodels==0.14.4 matplotlib==3.9.2 rapidfuzz==3.10.1 dateparser==1.2.0 ftfy==6.3.1


In [ ]:
# Helpers copied from Sherlock: the cleaning helpers the steps below use, and how a step is stored.
import duckdb
import re
import numpy as np
import pandas as pd


SENTINELS = frozenset({
    "", "n/a", "na", "null", "none", "nan", "?", "-", "--", "#n/a", "missing", "undefined", "nil",
})


_CURRENCY = re.compile(r"[$€£¥₹]|\b(?:usd|eur|gbp|jpy|inr|cad|aud|chf)\b", re.I)


_SPACES = re.compile(r"\s+")


_DATE_FORMATS = {
    "dmy": ("%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y", "%d/%m/%y", "%d %b %Y", "%d %B %Y", "%d-%b-%Y"),
    "mdy": ("%m/%d/%Y", "%m-%d-%Y", "%m/%d/%y", "%b %d, %Y", "%B %d, %Y", "%b %d %Y"),
    "ymd": ("%Y-%m-%d", "%Y/%m/%d", "%Y.%m.%d", "%Y%m%d"),
}


def _text(s: pd.Series) -> pd.Series:
    """Values as stripped strings, missing stays missing."""
    return s.where(s.isna(), s.astype(str).str.strip())


def _is_number(value) -> bool:
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False


def _comma_decimal(t: pd.Series, hint) -> bool:
    if hint is not None:
        h = str(hint).lower()
        return h in ("comma", "eu") or h[:2] in ("de", "fr", "es", "it", "nl", "pt", "ru", "pl", "tr", "da", "sv", "nb", "fi")
    sample = t.dropna()
    sample = sample[sample != ""].head(2000)
    if sample.empty:
        return False
    comma_last = sample.str.fullmatch(r"-?[\d.]*,\d{1,2}").mean()
    dot_last = sample.str.fullmatch(r"-?[\d,]*\.\d+").mean()
    return bool(comma_last > dot_last)


def extract_pattern(s: pd.Series, regex, group=1) -> pd.Series:
    """The `group` of the first match of `regex` in each value (missing where there is no match)."""
    return _text(s).astype("string").str.extract(regex, expand=True)[group - 1 if isinstance(group, int) else group].astype(object)


def fill_by_group(df: pd.DataFrame, col, by, stat="median") -> pd.Series:
    """`col` with missing values filled by the `stat` ("median", "mean" or "mode") of rows sharing
    `by` (a column or list of columns); groups with no value fall back to the whole column's."""
    keys = [by] if isinstance(by, str) else list(by)
    usable = df["_quarantine_reason"].isna() if "_quarantine_reason" in df else pd.Series(True, index=df.index)
    if stat == "mode":
        def mode(x):
            m = x.dropna().mode()
            return m.iloc[0] if len(m) else np.nan
        group = df[usable].groupby(keys, dropna=False)[col].agg(mode)
        overall = mode(df.loc[usable, col])
    else:
        group = df[usable].groupby(keys, dropna=False)[col].agg(stat)
        overall = df.loc[usable, col].agg(stat)
    filler = df[keys].merge(group.rename("_fill").reset_index(), on=keys, how="left")["_fill"]
    filler.index = df.index
    filled = df[col].fillna(filler)
    return filled.fillna(overall)


INT_TYPES = ("BIGINT", "INTEGER", "SMALLINT", "TINYINT", "HUGEINT", "UBIGINT", "UINTEGER")


def _q(name: str) -> str:
    return '"' + str(name).replace('"', '""') + '"'


def output_select(result, input_types: dict[str, str]) -> str:
    """The SELECT that writes `result`, casting columns back to their input type where pandas
    changed the type only in transit (nullable ints → float, DATE → timestamp, all-missing → object)."""
    import pandas as pd

    parts = []
    for column in result.columns:
        s = result[column]
        original = input_types.get(column)
        expr = _q(column)
        if column == "_row_id":
            expr = f"CAST({expr} AS BIGINT)"
        elif column == "_quarantine_reason":
            expr = f"CAST({expr} AS VARCHAR)"
        elif original is not None:
            present = s.dropna()
            if present.empty:
                expr = f"CAST({expr} AS {original})"
            elif original in INT_TYPES and s.dtype.kind == "f" and bool((present % 1 == 0).all()):
                expr = f"CAST({expr} AS {original})"
            elif original == "DATE" and s.dtype.kind == "M" and bool((present.dt.normalize() == present).all()):
                expr = f"CAST({expr} AS DATE)"
            elif original.startswith("TIMESTAMP") and s.dtype.kind == "M":
                expr = f"CAST({expr} AS {original})"
            elif original == "BOOLEAN" and s.dtype == object and present.map(lambda v: isinstance(v, bool)).all():
                expr = f"CAST({expr} AS BOOLEAN)"
            elif original == "VARCHAR" and s.dtype == object and pd.api.types.infer_dtype(present) == "string":
                expr = f"CAST({expr} AS VARCHAR)"
        parts.append(f"{expr} AS {_q(column)}")
    return "SELECT " + ", ".join(parts) + " FROM result_frame"


def keep_types(df, types):
    """`df` with each column cast to its DuckDB type in Sherlock's version (`types`: name → type),
    read back through DuckDB as the app reads every version."""
    cols = ", ".join(f"CAST({_q(c)} AS {t}) AS {_q(c)}" for c, t in types.items())
    return duckdb.sql(f"SELECT {cols} FROM df ORDER BY _row_id").df()


def store_step(result, types):
    """Store a cleaning step's `result` as Sherlock does and read it back: returns (df, types)."""
    rel = duckdb.sql(output_select(result, types).replace("FROM result_frame", "FROM result") + " ORDER BY _row_id")
    return rel.df(), {c: str(t) for c, t in zip(rel.columns, rel.types)}


In [ ]:
# Load: decode the raw file (utf-8) into a UTF-8 copy, then read it as Sherlock's import did.
from pathlib import Path

UTF8_PATH = Path(DATA_PATH).stem + ".utf8.csv"
Path(UTF8_PATH).write_text(Path(DATA_PATH).read_bytes().decode('utf-8'), encoding="utf-8", newline="")
raw = duckdb.sql(f"SELECT * FROM read_csv('{UTF8_PATH}', delim=',', header=true, skip=0, all_varchar=true, null_padding=true)").df()
raw.columns = ['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest']
raw.insert(0, "_quarantine_reason", None)
raw.insert(0, "_row_id", range(1, len(raw) + 1))
types = {"_row_id": "BIGINT", "_quarantine_reason": "VARCHAR", "pclass": "VARCHAR", "survived": "VARCHAR", "name": "VARCHAR", "sex": "VARCHAR", "age": "VARCHAR", "sibsp": "VARCHAR", "parch": "VARCHAR", "ticket": "VARCHAR", "fare": "VARCHAR", "cabin": "VARCHAR", "embarked": "VARCHAR", "boat": "VARCHAR", "body": "VARCHAR", "home.dest": "VARCHAR"}
df = keep_types(raw, types)
print(f"Loaded {len(df):,} rows and {len(df.columns) - 2} columns.")


## What stands out

- Survival rate was highest for female passengers, at 72.7%.
- Fare median was highest for pclass 1 passengers, at 60.00.
- Overall survival rate was 38.2%.

<details><summary>Evidence</summary>

```sql
SELECT "sex", AVG(CAST("survived" AS DOUBLE)) AS rate, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "sex" IS NOT NULL AND "survived" IS NOT NULL GROUP BY 1 ORDER BY 2 DESC, 1
LIMIT 2000
```

```sql
SELECT "pclass", MIN("fare") AS min, QUANTILE_CONT("fare", 0.25) AS q1, MEDIAN("fare") AS median, QUANTILE_CONT("fare", 0.75) AS q3, MAX("fare") AS max, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "pclass" IS NOT NULL AND "fare" IS NOT NULL GROUP BY 1 ORDER BY 1
LIMIT 2000
```

```sql
SELECT AVG(CAST("survived" AS DOUBLE)) AS rate, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "survived" IS NOT NULL
LIMIT 2000
```

</details>

### Step 1 · Typed 7 columns from text: pclass, survived, age, sibsp, parch, fare and 1 more

By automatic, committed.  
1,309 rows in, 1,309 out · 0 cells changed.

- **Which columns hold numbers, true/false values or dates?** pclass, survived, age, sibsp, parch, fare, body. at least 99% of each column's values convert without loss

In [ ]:
rel = duckdb.sql("""
SELECT
  _row_id,
  coalesce(_quarantine_reason, CASE WHEN CASE WHEN lower(trim("pclass")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("pclass") END IS NOT NULL AND CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("pclass")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("pclass") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("pclass")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("pclass") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("pclass")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("pclass") END AS BIGINT) END IS NULL THEN 'type: pclass' WHEN CASE WHEN lower(trim("survived")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("survived") END IS NOT NULL AND CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("survived")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("survived") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("survived")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("survived") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("survived")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("survived") END AS BIGINT) END IS NULL THEN 'type: survived' WHEN CASE WHEN lower(trim("age")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("age") END IS NOT NULL AND CASE WHEN NOT regexp_matches(CASE WHEN lower(trim("age")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("age") END, '^[+-]?0[0-9]') AND length(trim(regexp_replace(regexp_replace(CASE WHEN lower(trim("age")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("age") END, '[eE].*$', ''), '[^0-9]', '', 'g'), '0')) <= 15 THEN TRY_CAST(CASE WHEN lower(trim("age")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("age") END AS DOUBLE) END IS NULL THEN 'type: age' WHEN CASE WHEN lower(trim("sibsp")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("sibsp") END IS NOT NULL AND CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("sibsp")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("sibsp") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("sibsp")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("sibsp") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("sibsp")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("sibsp") END AS BIGINT) END IS NULL THEN 'type: sibsp' WHEN CASE WHEN lower(trim("parch")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("parch") END IS NOT NULL AND CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("parch")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("parch") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("parch")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("parch") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("parch")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("parch") END AS BIGINT) END IS NULL THEN 'type: parch' WHEN CASE WHEN lower(trim("fare")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("fare") END IS NOT NULL AND CASE WHEN NOT regexp_matches(CASE WHEN lower(trim("fare")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("fare") END, '^[+-]?0[0-9]') AND length(trim(regexp_replace(regexp_replace(CASE WHEN lower(trim("fare")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("fare") END, '[eE].*$', ''), '[^0-9]', '', 'g'), '0')) <= 15 THEN TRY_CAST(CASE WHEN lower(trim("fare")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("fare") END AS DOUBLE) END IS NULL THEN 'type: fare' WHEN CASE WHEN lower(trim("body")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("body") END IS NOT NULL AND CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("body")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("body") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("body")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("body") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("body")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("body") END AS BIGINT) END IS NULL THEN 'type: body' END) AS _quarantine_reason,
  CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("pclass")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("pclass") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("pclass")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("pclass") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("pclass")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("pclass") END AS BIGINT) END AS "pclass",
  CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("survived")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("survived") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("survived")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("survived") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("survived")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("survived") END AS BIGINT) END AS "survived",
  "name",
  "sex",
  CASE WHEN NOT regexp_matches(CASE WHEN lower(trim("age")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("age") END, '^[+-]?0[0-9]') AND length(trim(regexp_replace(regexp_replace(CASE WHEN lower(trim("age")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("age") END, '[eE].*$', ''), '[^0-9]', '', 'g'), '0')) <= 15 THEN TRY_CAST(CASE WHEN lower(trim("age")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("age") END AS DOUBLE) END AS "age",
  CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("sibsp")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("sibsp") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("sibsp")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("sibsp") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("sibsp")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("sibsp") END AS BIGINT) END AS "sibsp",
  CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("parch")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("parch") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("parch")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("parch") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("parch")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("parch") END AS BIGINT) END AS "parch",
  "ticket",
  CASE WHEN NOT regexp_matches(CASE WHEN lower(trim("fare")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("fare") END, '^[+-]?0[0-9]') AND length(trim(regexp_replace(regexp_replace(CASE WHEN lower(trim("fare")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("fare") END, '[eE].*$', ''), '[^0-9]', '', 'g'), '0')) <= 15 THEN TRY_CAST(CASE WHEN lower(trim("fare")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("fare") END AS DOUBLE) END AS "fare",
  "cabin",
  "embarked",
  "boat",
  CASE WHEN CAST(TRY_CAST(CASE WHEN lower(trim("body")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("body") END AS BIGINT) AS VARCHAR) = regexp_replace(CASE WHEN lower(trim("body")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("body") END, '^[+]', '') THEN TRY_CAST(CASE WHEN lower(trim("body")) IN ('', '#n/a', '-', '--', '?', 'missing', 'n/a', 'na', 'nan', 'nil', 'none', 'null', 'undefined') THEN NULL ELSE trim("body") END AS BIGINT) END AS "body",
  "home.dest"
FROM df
ORDER BY _row_id
""")
types = {c: str(t) for c, t in zip(rel.columns, rel.types)}
df = rel.df()


### Step 2 · Median by title, pclass, and sex (age)

By the Cleaner, committed (confidence 0.90).  
1,309 rows in, 1,309 out · 263 cells changed.

- **How to fix missing in age?** Median by title, pclass, and sex. Age was not recorded for some passengers, particularly in third class, leading to missing values in the historical dataset.

In [ ]:
result = df.copy()
result['_title'] = extract_pattern(result['name'], r',\s*([A-Za-z]+)\.', group=1)
result['age'] = fill_by_group(result, 'age', by=['_title', 'pclass', 'sex'], stat='median')
result = result.drop(columns=['_title'])
df = result
df, types = store_step(df, types)


### Step 3 · Median by pclass (fare)

By the Cleaner, committed (confidence 0.93).  
1,309 rows in, 1,309 out · 1 cells changed.

- **How to fix missing in fare?** Median by pclass. One passenger's fare was not recorded in the original manifest, likely due to a clerical omission or because the ticket was complimentary or handled separately.

In [ ]:
result = df.copy()
result['fare'] = fill_by_group(result, 'fare', by='pclass', stat='median')
df = result
df, types = store_step(df, types)


In [ ]:
# Check: the rebuilt DataFrame matches Sherlock's version 4.
expected = {"_row_id": "BIGINT", "_quarantine_reason": "VARCHAR", "pclass": "BIGINT", "survived": "BIGINT", "name": "VARCHAR", "sex": "VARCHAR", "age": "DOUBLE", "sibsp": "BIGINT", "parch": "BIGINT", "ticket": "VARCHAR", "fare": "DOUBLE", "cabin": "VARCHAR", "embarked": "VARCHAR", "boat": "VARCHAR", "body": "BIGINT", "home.dest": "VARCHAR"}
assert len(df) == 1309, f"expected 1,309 rows, got {len(df):,}"
assert list(df.columns) == list(expected), "the columns differ from the version"
assert types == expected, "a column's type differs from the version"
print("Rebuilt version 4: 1,309 rows × 14 columns, matching Sherlock.")


## The dashboard

Each widget's query runs on the rebuilt data; each analysis runs its reproduction code.

In [ ]:
# Charts: one function draws every widget the way the dashboard does (greys, the point in the accent).
import matplotlib.pyplot as plt
from IPython.display import display

GREY, LIGHT, ACCENT = "#55575c", "#a9a69f", "#2f6b62"


def plot(result, view, x=None, y=None, series=None, title=None, point=None):
    cols = list(result.columns)
    x = x if x in cols else cols[0]
    y = y if y in cols else (cols[1] if len(cols) > 1 else cols[0])
    if view == "kpi":
        print(f"{title or y}: {result[y].iloc[0]}")
        return
    if view == "table" or result.empty:
        display(result.head(50))
        return
    fig, ax = plt.subplots(figsize=(7, 3.4))
    if view in ("line", "area"):
        ax.plot(result[x], result[y], color=GREY)
        if view == "area":
            ax.fill_between(result[x], result[y], color=LIGHT, alpha=0.4)
    elif view == "scatter":
        ax.scatter(result[x], result[y], s=8, color=GREY, alpha=0.6)
    elif view == "heatmap" and series in cols:
        grid = result.pivot_table(index=series, columns=x, values=y, aggfunc="first")
        ax.imshow(grid.values, cmap="Greys", aspect="auto")
        ax.set_xticks(range(len(grid.columns)), [str(c) for c in grid.columns])
        ax.set_yticks(range(len(grid.index)), [str(i) for i in grid.index])
    elif view == "box" and {"q1", "median", "q3"} <= set(cols):
        stats = [{"label": str(r[x]), "whislo": r.get("lo", r.get("min")), "q1": r["q1"], "med": r["median"],
                  "q3": r["q3"], "whishi": r.get("hi", r.get("max")), "fliers": []} for _, r in result.iterrows()]
        ax.bxp(stats, showfliers=False)
    elif view in ("importance", "coefficients"):
        ax.barh(result[x].astype(str)[::-1], result[y][::-1], color=GREY)
    else:  # bar, histogram, and pie, ring and gauge drawn as bars
        colors = [ACCENT if point is not None and str(v) == str(point) else GREY for v in result[x]]
        ax.bar(result[x].astype(str), result[y], color=colors)
    ax.set_title(title or "", loc="left", fontsize=11)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    plt.show()


## Total passengers

Rows: all rows · view: kpi

In [ ]:
# The dashboard's global filter isn't applied here; the widget's own rows are (in the WHERE clause).
t = df
result = duckdb.sql("""
SELECT COUNT(*) AS n_rows FROM t WHERE _quarantine_reason IS NULL AND (1=1)
LIMIT 2000
""").df()
result


n_rows
1309


In [ ]:
plot(result, 'kpi', title='Total passengers')


### Total passengers numbered 1,309.

The dataset includes 1,309 records.


<details><summary>Evidence</summary>

```sql
SELECT COUNT(*) AS n_rows FROM t WHERE _quarantine_reason IS NULL AND (1=1)
LIMIT 2000
```

```sql
SELECT COUNT(*) FROM t WHERE _quarantine_reason IS NULL AND (1=1)
```

</details>

## Survival rate

Rows: all rows · view: kpi

In [ ]:
# The dashboard's global filter isn't applied here; the widget's own rows are (in the WHERE clause).
t = df
result = duckdb.sql("""
SELECT AVG(CAST("survived" AS DOUBLE)) AS rate, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "survived" IS NOT NULL
LIMIT 2000
""").df()
result


rate,n
0.381971,1309


In [ ]:
plot(result, 'kpi', title='Survival rate')


### Overall survival rate was 38.2%.

This figure reflects the proportion of passengers who survived the event. It shows that a minority of passengers survived. A related widget breaks down survival by class and sex.


Related: What drives survived? · Survival by sex · Survival by class and sex

<details><summary>Evidence</summary>

```sql
SELECT AVG(CAST("survived" AS DOUBLE)) AS rate, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "survived" IS NOT NULL
LIMIT 2000
```

</details>

## Average fare

Rows: all rows · view: kpi

In [ ]:
# The dashboard's global filter isn't applied here; the widget's own rows are (in the WHERE clause).
t = df
result = duckdb.sql("""
SELECT AVG("fare") AS value, COUNT("fare") AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1)
LIMIT 2000
""").df()
result


value,n
33.276193,1309


In [ ]:
plot(result, 'kpi', title='Average fare')


### The average fare was 33.28.

This figure is based on data from 1,309 passengers. It aligns with the mid-range observed in the fare distribution by class widget.


Related: Fare distribution by class · What drives survived?

<details><summary>Evidence</summary>

```sql
SELECT AVG("fare") AS value, COUNT("fare") AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1)
LIMIT 2000
```

```sql
SELECT COUNT(*) FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "fare" IS NOT NULL
```

</details>

## Average age

Rows: all rows · view: kpi

In [ ]:
# The dashboard's global filter isn't applied here; the widget's own rows are (in the WHERE clause).
t = df
result = duckdb.sql("""
SELECT AVG("age") AS value, COUNT("age") AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1)
LIMIT 2000
""").df()
result


value,n
29.272473,1309


In [ ]:
plot(result, 'kpi', title='Average age')


### Average age 29.27.

This widget summarizes the typical age across the dataset.  For more detail, see the Age distribution by class widget.  The Age distribution widget shows how ages vary.


Related: Age distribution by class · Age distribution

<details><summary>Evidence</summary>

```sql
SELECT AVG("age") AS value, COUNT("age") AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1)
LIMIT 2000
```

</details>

## Fare distribution by class

Rows: all rows · view: box

In [ ]:
# The dashboard's global filter isn't applied here; the widget's own rows are (in the WHERE clause).
t = df
result = duckdb.sql("""
SELECT "pclass", MIN("fare") AS min, QUANTILE_CONT("fare", 0.25) AS q1, MEDIAN("fare") AS median, QUANTILE_CONT("fare", 0.75) AS q3, MAX("fare") AS max, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "pclass" IS NOT NULL AND "fare" IS NOT NULL GROUP BY 1 ORDER BY 1
LIMIT 2000
""").df()
result


pclass,min,q1,median,q3,max,n
1,0.0,30.6958,60.0000,107.6625,512.3292,323
2,0.0,13.0000,15.0458,26.0000,73.5000,277
3,0.0,7.7500,8.0500,15.2458,69.5500,709


In [ ]:
plot(result, 'box', x='pclass', y='median', title='Fare distribution by class', point='1')


### Median fare was highest for pclass 1 at 60.00.

Median fare was lowest for pclass 3 at 8.05.


Related: Average fare · Age distribution by class · Survival by class and sex

<details><summary>Evidence</summary>

```sql
SELECT "pclass", MIN("fare") AS min, QUANTILE_CONT("fare", 0.25) AS q1, MEDIAN("fare") AS median, QUANTILE_CONT("fare", 0.75) AS q3, MAX("fare") AS max, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "pclass" IS NOT NULL AND "fare" IS NOT NULL GROUP BY 1 ORDER BY 1
LIMIT 2000
```

</details>

## Survival by class and sex

Rows: all rows · view: heatmap

In [ ]:
# The dashboard's global filter isn't applied here; the widget's own rows are (in the WHERE clause).
t = df
result = duckdb.sql("""
SELECT "pclass", "sex", AVG(CAST("survived" AS DOUBLE)) AS rate, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "pclass" IS NOT NULL AND "sex" IS NOT NULL AND "survived" IS NOT NULL GROUP BY 1, 2 ORDER BY 1, 2
LIMIT 2000
""").df()
result


pclass,sex,rate,n
1,female,0.965278,144
1,male,0.340782,179
2,female,0.886792,106
2,male,0.146199,171
3,female,0.490741,216
3,male,0.152130,493


In [ ]:
plot(result, 'heatmap', x='pclass', y='rate', series='sex', title='Survival by class and sex', point='1')


### For female passengers in pclass 1 class, survival was 96.5%.

The widget displays survival rates broken down by pclass and sex.


Related: What drives survived? · Survival rate · Survival by sex · Fare distribution by class

<details><summary>Evidence</summary>

```sql
SELECT "pclass", "sex", AVG(CAST("survived" AS DOUBLE)) AS rate, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "pclass" IS NOT NULL AND "sex" IS NOT NULL AND "survived" IS NOT NULL GROUP BY 1, 2 ORDER BY 1, 2
LIMIT 2000
```

</details>

## Survival by sex

Rows: all rows · view: bar

In [ ]:
# The dashboard's global filter isn't applied here; the widget's own rows are (in the WHERE clause).
t = df
result = duckdb.sql("""
SELECT "sex", AVG(CAST("survived" AS DOUBLE)) AS rate, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "sex" IS NOT NULL AND "survived" IS NOT NULL GROUP BY 1 ORDER BY 2 DESC, 1
LIMIT 2000
""").df()
result


sex,rate,n
female,0.727468,466
male,0.190985,843


In [ ]:
plot(result, 'bar', x='sex', y='rate', title='Survival by sex', point='female')


### Survival was highest for female passengers, at 72.7%.

The rate for male passengers was 19.1%. Overall survival across all passengers was 38.2%. The difference between sex groups is unlikely to be chance, with p<0.001. The association between sex and survived is moderate, with an effect size of 0.527.


Related: What drives survived? · Survival rate · Survival by class and sex

<details><summary>Evidence</summary>

```sql
SELECT "sex", AVG(CAST("survived" AS DOUBLE)) AS rate, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "sex" IS NOT NULL AND "survived" IS NOT NULL GROUP BY 1 ORDER BY 2 DESC, 1
LIMIT 2000
```

</details>

## Age distribution by class

Rows: all rows · view: box

In [ ]:
# The dashboard's global filter isn't applied here; the widget's own rows are (in the WHERE clause).
t = df
result = duckdb.sql("""
SELECT "pclass", MIN("age") AS min, QUANTILE_CONT("age", 0.25) AS q1, MEDIAN("age") AS median, QUANTILE_CONT("age", 0.75) AS q3, MAX("age") AS max, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "pclass" IS NOT NULL AND "age" IS NOT NULL GROUP BY 1 ORDER BY 1
LIMIT 2000
""").df()
result


pclass,min,q1,median,q3,max,n
1,0.9167,30.0,41.5,48.0,80.0,323
2,0.6667,22.0,29.0,36.0,70.0,277
3,0.1667,18.0,26.0,29.0,74.0,709


In [ ]:
plot(result, 'box', x='pclass', y='median', title='Age distribution by class', point='1')


### Median age varies across passenger class.

pclass 1 passengers have the highest median age, at 41.50 years. pclass 3 passengers have the lowest median age, at 26.00 years. The analysis includes 1,309 passengers.


Related: Age distribution · Fare distribution by class · Average age

<details><summary>Evidence</summary>

```sql
SELECT "pclass", MIN("age") AS min, QUANTILE_CONT("age", 0.25) AS q1, MEDIAN("age") AS median, QUANTILE_CONT("age", 0.75) AS q3, MAX("age") AS max, COUNT(*) AS n FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "pclass" IS NOT NULL AND "age" IS NOT NULL GROUP BY 1 ORDER BY 1
LIMIT 2000
```

```sql
SELECT COUNT(*) FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "pclass" IS NOT NULL AND "age" IS NOT NULL
```

</details>

## Age distribution

Rows: all rows · view: histogram

In [ ]:
# The dashboard's global filter isn't applied here; the widget's own rows are (in the WHERE clause).
t = df
result = duckdb.sql("""
WITH b AS (SELECT MIN("age") AS lo, (MAX("age") - MIN("age")) / 20 AS w FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "age" IS NOT NULL) SELECT b.lo + b.w * COALESCE(LEAST(FLOOR(("age" - b.lo) / NULLIF(b.w, 0)), 19), 0) AS bin_start, COUNT(*) AS n FROM t, b WHERE _quarantine_reason IS NULL AND (1=1) AND "age" IS NOT NULL GROUP BY 1 ORDER BY 1
LIMIT 2000
""").df()
result


bin_start,n
0.166700,51
4.158365,29
8.150030,22
12.141695,40
16.133360,163
20.125025,160
24.116690,265
28.108355,153
32.100020,97
36.091685,64


In [ ]:
plot(result, 'histogram', x='bin_start', y='n', title='Age distribution')


### The age distribution shows a median age of 26.00 years, with an average age of 29.27 years.

The lowest age is 0.167 years, while the highest age is 80.00 years. The outlier share is 3.1%.


Related: Age distribution by class · Average age

<details><summary>Evidence</summary>

```sql
SELECT MEDIAN("age") FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "age" IS NOT NULL
```

```sql
SELECT AVG("age") FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "age" IS NOT NULL
```

```sql
SELECT MIN("age") FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "age" IS NOT NULL
```

```sql
SELECT MAX("age") FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "age" IS NOT NULL
```

```sql
SELECT AVG(CASE WHEN "age" < q1 - 1.5 * (q3 - q1) OR "age" > q3 + 1.5 * (q3 - q1) THEN 1 ELSE 0 END) FROM (SELECT "age", QUANTILE_CONT("age", 0.25) OVER () AS q1, QUANTILE_CONT("age", 0.75) OVER () AS q3 FROM t WHERE _quarantine_reason IS NULL AND (1=1) AND "age" IS NOT NULL)
```

</details>

## What drives survived?

What drives **survived** · Rows: all rows · Auto picked Logistic regression by 5-fold cross-validation (seed 42), keeping rows that share a `ticket` in one fold.

**sex alone does nearly as well (78.1%): it carries the signal, and the other columns add only 0.1 points on held-out rows.**

Left out, with the reason:

- `name`: ID-like: unique per row
- `boat`: records the outcome (leakage): near-perfect predictor of survived (0.98)
- `body`: records the outcome (leakage): near-perfect predictor of survived (1.00)

In [ ]:
# Reproduces Sherlock run run_01M5TQGJ3A1R99MSD16NEA70HG (driver analysis on version 4).
# Auto tried: Gradient boosting (accuracy 0.763), Random forest (accuracy 0.768). The winner, Logistic regression, is reproduced here.
import duckdb

import warnings


import numpy as np


import pandas as pd


from sklearn.compose import ColumnTransformer


from sklearn.dummy import DummyClassifier, DummyRegressor


from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor


from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor


from sklearn.impute import SimpleImputer


from sklearn.inspection import permutation_importance


from sklearn.linear_model import LogisticRegression, Ridge


from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score, roc_auc_score


from sklearn.model_selection import GroupKFold, KFold, StratifiedGroupKFold, StratifiedKFold, train_test_split


from sklearn.pipeline import Pipeline


from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder


from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor


FOLDS = 5


def sample(data, target, task, n, seed):
    """`n` of the rows, drawn with the seed and back in row order. For categories the draw is stratified
    by the outcome, so each value keeps its share (TRD §6.7)."""
    if len(data) <= n:
        return data
    if task == "classification":
        data, _ = train_test_split(data, train_size=n, stratify=data[target].astype(str), random_state=seed)
    else:
        data = data.sample(n=n, random_state=seed)
    return data.sort_values("_row_id")


def prepare(data, target, numeric, categorical, high_card):
    """X and y: numbers as floats (missing stays missing), categories as text with "(missing)"."""
    X = pd.DataFrame(index=data.index)
    for c in numeric:
        X[c] = pd.to_numeric(data[c], errors="coerce").astype(float)
    for c in categorical + high_card:
        X[c] = data[c].astype(str).where(data[c].notna(), "(missing)")
    y = data[target]
    if y.dtype == object or str(y.dtype) == "bool":
        y = y.astype(str)
    return X, y.reset_index(drop=True).set_axis(X.index)


def preprocessor(numeric, categorical, high_card, task, seed):
    """Imputation, scaling and encoding, fitted inside each fold (TRD §7.5 step 5)."""
    parts = []
    if numeric:
        parts.append(("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric))
    if categorical:
        parts.append(("categorical", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10,
                                                   sparse_output=False), categorical))
    if high_card:
        parts.append(("high_cardinality", TargetEncoder(target_type="continuous" if task == "regression" else "auto",
                                                        random_state=seed), high_card))
    return ColumnTransformer(parts, remainder="drop")


def make_model(name, task, seed):
    if task == "classification":
        return {"logistic_regression": LogisticRegression(C=1.0, max_iter=2000),
                "gradient_boosting": HistGradientBoostingClassifier(random_state=seed),
                "random_forest": RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=1),
                "majority": DummyClassifier(strategy="most_frequent"),
                "one_column_rule": DecisionTreeClassifier(max_depth=1, random_state=seed)}[name]
    return {"ridge": Ridge(alpha=1.0),
            "gradient_boosting": HistGradientBoostingRegressor(random_state=seed),
            "random_forest": RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=1),
            "majority": DummyRegressor(strategy="mean"),
            "one_column_rule": DecisionTreeRegressor(max_depth=1, random_state=seed)}[name]


def group_labels(data, column):
    """One label per row for keeping rows that share a value of `column` in one fold. A blank is a
    group of its own."""
    labels = data[column].astype(str).to_numpy(dtype=object)
    blank = data[column].isna().to_numpy()
    labels[blank] = [f"(blank {i})" for i in range(int(blank.sum()))]
    return labels


def folds(X, y, task, seed, n_folds=FOLDS, groups=None):
    """The cross-validation splits and whether they keep groups together (TRD §7.5 step 4). With
    `groups`, the rows of one group (a shared ticket, account or household) stay in one fold, so a
    held-out row's relatives aren't among its training rows; for categories the folds stay
    stratified. When grouping leaves a fold without one of the outcomes, or there are fewer groups
    than folds, the folds are the plain ones."""
    if groups is not None and len(set(groups)) >= n_folds:
        split = StratifiedGroupKFold(n_folds, shuffle=True, random_state=seed) if task == "classification" else GroupKFold(n_folds)
        out = list(split.split(X, y, groups))
        if task == "regression" or all(y.iloc[test].nunique() == y.nunique() for _, test in out):
            return out, True
    split = (StratifiedKFold(n_folds, shuffle=True, random_state=seed) if task == "classification"
             else KFold(n_folds, shuffle=True, random_state=seed))
    return list(split.split(X, y)), False


def metric_name(task, y):
    if task == "regression":
        return "r2"
    return "accuracy" if y.nunique() == 2 else "macro_f1"


def score(metric, y_true, y_pred):
    if metric == "accuracy":
        return float(accuracy_score(y_true, y_pred))
    if metric == "macro_f1":
        return float(f1_score(y_true, y_pred, average="macro"))
    return float(r2_score(y_true, y_pred))


def pipeline(name, numeric, categorical, high_card, task, seed):
    return Pipeline([("prepare", preprocessor(numeric, categorical, high_card, task, seed)),
                     ("model", make_model(name, task, seed))])


def evaluate(name, X, y, task, seed, numeric, categorical, high_card, splits=None):
    """Cross-validated scores of one model: {metric, folds, mean, auc?, mae?} plus the fitted
    per-fold pipelines (for importance)."""
    warnings.filterwarnings("ignore", message="Unknown solver options")  # scikit-learn 1.5 with a newer SciPy
    metric = metric_name(task, y)
    splits = splits or folds(X, y, task, seed)[0]
    out = {"metric": metric, "folds": [], "fitted": []}
    aucs, maes = [], []
    for train, test in splits:
        p = pipeline(name, numeric, categorical, high_card, task, seed)
        p.fit(X.iloc[train], y.iloc[train])
        pred = p.predict(X.iloc[test])
        out["folds"].append(score(metric, y.iloc[test], pred))
        out["fitted"].append((p, test))
        if metric == "accuracy" and hasattr(p, "predict_proba") and name != "majority":
            proba = p.predict_proba(X.iloc[test])[:, 1]
            aucs.append(float(roc_auc_score(y.iloc[test], proba)))
        if task == "regression":
            maes.append(float(mean_absolute_error(y.iloc[test], pred)))
    out["mean"] = float(np.mean(out["folds"]))
    if aucs:
        out["auc"] = float(np.mean(aucs))
    if maes:
        out["mae"] = float(np.mean(maes))
    return out


def baselines(X, y, task, seed, numeric, categorical, high_card, splits):
    """The majority (or mean) and the best one-column rule (a depth-one tree on each column)."""
    first = evaluate("majority", X, y, task, seed, numeric, categorical, high_card, splits)
    label = "everyone gets the most common outcome" if task == "classification" else "predict the average"
    rows = [{"name": label, "metric": first["metric"], "value": first["mean"]}]
    best = None
    for col in numeric + categorical + high_card:
        one = evaluate("one_column_rule", X[[col]], y, task, seed,
                       [col] if col in numeric else [], [col] if col in categorical else [],
                       [col] if col in high_card else [], splits)
        if best is None or one["mean"] > best[1]:
            best = (col, one["mean"])
    if best is not None:
        rows.append({"name": f"best one-column rule: {best[0]}", "metric": first["metric"], "value": best[1], "column": best[0]})
    return rows


def importance(fitted, X, y, metric, seed):
    """Permutation importance on each held-out fold (10 repeats), averaged; 1.0 is the strongest."""
    scoring = {"accuracy": "accuracy", "macro_f1": "f1_macro", "r2": "r2"}[metric]
    total = pd.Series(0.0, index=X.columns)
    for p, test in fitted:
        r = permutation_importance(p, X.iloc[test], y.iloc[test], n_repeats=10, random_state=seed, scoring=scoring)
        total += pd.Series(r.importances_mean, index=X.columns)
    raw = (total / len(fitted)).clip(lower=0)
    top = raw.max()
    table = pd.DataFrame({"feature": raw.index, "importance": (raw / top if top > 0 else raw).round(4), "raw": raw.round(6)})
    return table.sort_values(["importance", "feature"], ascending=[False, True]).reset_index(drop=True)


def run(data, target, numeric, categorical, high_card, task, model, seed=42, n_folds=FOLDS, group=None):
    """Train and score `model` the way Sherlock did: the same folds, baselines and importance."""
    X, y = prepare(data, target, numeric, categorical, high_card)
    splits, grouped = folds(X, y, task, seed, n_folds, group_labels(data, group) if group else None)
    result = evaluate(model, X, y, task, seed, numeric, categorical, high_card, splits)
    base = baselines(X, y, task, seed, numeric, categorical, high_card, splits)
    imp = importance(result["fitted"], X, y, result["metric"], seed)
    sd = float(np.std(result["folds"], ddof=1)) if len(result["folds"]) > 1 else 0.0
    half = float(1.96 * sd / np.sqrt(len(result["folds"])))
    metrics = {"metric": result["metric"], "value": round(result["mean"], 6),
               "ci": [round(result["mean"] - half, 6), round(result["mean"] + half, 6)],
               "folds": [round(f, 6) for f in result["folds"]]}
    for k in ("auc", "mae"):
        if k in result:
            metrics[k] = round(result[k], 6)
    out = {"metrics": metrics, "baselines": base, "importance": imp, "rows": int(len(X)), "grouped_by": group if grouped else None}
    if task == "classification":
        out["oof"] = out_of_fold(result["fitted"], X, y, data)
    return out


def out_of_fold(fitted, X, y, data):
    """Each row's prediction from the fold that held it out: `_row_id`, actual, predicted and, for
    two classes, the probability of the second class (sorted order)."""
    frames = []
    for p, test in fitted:
        part = pd.DataFrame({"actual": y.iloc[test].astype(str).values,
                             "predicted": pd.Series(p.predict(X.iloc[test])).astype(str).values}, index=X.index[test])
        if y.nunique() == 2:
            part["probability"] = p.predict_proba(X.iloc[test])[:, 1]
        frames.append(part)
    oof = pd.concat(frames).sort_index()
    if "_row_id" in data.columns:
        oof.insert(0, "_row_id", data.loc[oof.index, "_row_id"].astype(int).values)
    return oof.reset_index(drop=True)


def reproduce(df, seed=42):
    """What drives survived: the same rows, features, folds and model as the app."""
    t = df
    data = duckdb.sql('SELECT * FROM t WHERE _quarantine_reason IS NULL AND (1=1) ORDER BY _row_id').df()  # the run's rows (quarantined rows excluded)
    data = data[data['survived'].notna()]
    # Features used: pclass, sex, age, sibsp, parch, ticket, fare, cabin, embarked, home.dest
    # Left out, with the reason:
#   name: ID-like: unique per row
#   boat: records the outcome (leakage): near-perfect predictor of survived (0.98)
#   body: records the outcome (leakage): near-perfect predictor of survived (1.00)
    # Rows that share a ticket stay in one fold, so a held-out row's group isn't in its training rows.
    return run(data, 'survived', ['age', 'fare'], ['pclass', 'sex', 'sibsp', 'parch', 'embarked'], ['ticket', 'cabin', 'home.dest'], 'classification', 'logistic_regression', seed,
               n_folds=5, group='ticket')


from IPython.display import display

run_result = reproduce(df)
m = run_result["metrics"]
print(f"{m['metric']}: {m['value']:.3f} (95% interval {m['ci'][0]:.3f} to {m['ci'][1]:.3f}) on {run_result['rows']:,} rows")
display(pd.DataFrame(run_result["baselines"]))
run_result["importance"]


accuracy: 0.782 (95% interval 0.753 to 0.810) on 1,309 rows


name,metric,value,column
everyone gets the most common outcome,accuracy,0.619153,None
best one-column rule: sex,accuracy,0.781067,sex


feature,importance
sex,1.0000
pclass,0.0998
age,0.0368
cabin,0.0008
sibsp,0.0008
embarked,0.0000
fare,0.0000
home.dest,0.0000
parch,0.0000
ticket,0.0000


In [ ]:
plot(run_result["importance"].head(15), "importance", x="feature", y="importance", title='What drives survived?')


### The top driver of survival is sex with importance 1.00.

The second most important factor is pclass with importance 0.0998. The third most important factor is age with importance 0.0368. The model achieves 78.2% accuracy on held‑out rows, which is 0.06 points above the best single‑column rule (sex) that scores 78.1%.


Related: Survival by sex · Survival rate · Survival by class and sex · Average fare

## From the Flow canvas

1 card written from other cards or from chat answers. Each number in them is a fact computed above, with its query under Evidence.

### These widgets describe the Titanic passenger dataset.

All widgets are based on 1,309 rows. They report the total passenger count, the survival rate, and the average fare.


<details><summary>Evidence</summary>

```sql
SELECT COUNT(*) FROM t WHERE _quarantine_reason IS NULL AND (1=1)
```

</details>

## How to continue

- `df` holds the cleaned data and `types` its column types; add cells below to keep exploring.
- Query `df` with SQL as the widgets do: `t = df; duckdb.sql("SELECT … FROM t").df()`.
- The helpers cell near the top holds the cleaning functions the steps used, with docstrings.
- Sherlock: https://sherlock-anw3.onrender.com